In [1]:
import networkx as nx
import numpy as np
from bngenerator import *
from matplotlib import pyplot as plt
from pgmpy.readwrite.XMLBeliefNetwork import XBNReader, XBNWriter
from pgmpy.readwrite import BIFWriter, BIFReader
from pgmpy.models import BayesianModel
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination, ApproxInference, BeliefPropagation
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BDeuScore, K2Score, BicScore
from pgmpy.metrics import structure_score
from pgmpy.utils import get_example_model
from pgmpy.estimators import ScoreCache
from pgmpy.inference.CausalInference import CausalInference
import random
import itertools
import numpy as np
import math
from same_decision_probability_calculation import *
from utils import *
from monte_carlo_sdp import *
import os
import glob
import time

In [2]:
def parse_bn_filename(filename):
    base = os.path.basename(filename).replace('.bif', '').replace('bn_', '')
    parts = base.split('_')
    n_nodes = int(parts[0].replace('n', ''))
    w_weight = int(parts[1].replace('w', ''))
    density = parts[2]
    rigidity = parts[3]
    return n_nodes, density, rigidity

In [3]:
n, d, r = parse_bn_filename('bn_n60_w6_medium_fuzzy.bif')
print(n,d,r)

60 medium fuzzy


# One case test

In [19]:
model = BIFReader('/home/joao/Desktop/UFMG/PhD/code/explaining-BN/src/generated_bif_files/bn_n80_w12_dense_det.bif').get_model()

In [5]:
model.nodes()

NodeView(('X0', 'X1', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X2', 'X20', 'X21', 'X22', 'X23', 'X24', 'X25', 'X26', 'X27', 'X28', 'X29', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9'))

In [6]:
def select_optimal_target_node(bn):
    """
    Selects a target node deeply embedded in the network (highest degree).
    Highly connected nodes provide a smooth, responsive probability gradient 
    for the Hill Climber, making extreme SDP values (0.90+) reachable.
    """
    best_node = None
    max_degree = -1
    
    for node in bn.nodes():
        # Ensure it's a binary node
        if len(bn.get_cpds(node).state_names[node]) != 2:
            continue
            
        degree = len(bn.get_parents(node)) + len(bn.get_children(node))
        if degree > max_degree:
            max_degree = degree
            best_node = node
            
    # Fallback if no binary nodes exist (rare)
    if best_node is None:
        return random.choice(list(bn.nodes()))
        
    return best_node

In [20]:
all_nodes = model.nodes()
all_nodes = list(all_nodes)
#target = random.choice(all_nodes)
target = select_optimal_target_node(model)
target_states = model.get_cpds(target).state_names[target]
target_value = target_states[1] if len(target_states) > 1 else target_states[0]
H_RATIO = 0.25
available_nodes = [n for n in all_nodes if n != target]
n_hidden = max(1, int(len(available_nodes) * H_RATIO))
hidden_vars = random.sample(available_nodes, n_hidden)
evidence_vars = [n for n in available_nodes if n not in hidden_vars]

In [21]:
n_hidden

19

In [22]:
model.get_cpds(target).state_names[target]

['0', '1']

In [9]:
print(model.get_cpds(target))

+-------+--------------------+-----+-----------------------+
| X1    | X1(0)              | ... | X1(1)                 |
+-------+--------------------+-----+-----------------------+
| X6    | X6(0)              | ... | X6(1)                 |
+-------+--------------------+-----+-----------------------+
| X7    | X7(0)              | ... | X7(1)                 |
+-------+--------------------+-----+-----------------------+
| X8    | X8(0)              | ... | X8(1)                 |
+-------+--------------------+-----+-----------------------+
| X14   | X14(0)             | ... | X14(1)                |
+-------+--------------------+-----+-----------------------+
| X15   | X15(0)             | ... | X15(1)                |
+-------+--------------------+-----+-----------------------+
| X3(0) | 0.3376503334361485 | ... | 8.257533592792576e-06 |
+-------+--------------------+-----+-----------------------+
| X3(1) | 0.6623496665638515 | ... | 0.9999917424664072    |
+-------+---------------

In [26]:
patients = build_experimental_dataset(model, target, '1', 0.5, evidence_vars, max_attempts=10, buckets=[0.55, 0.70, 0.80, 0.9, 1])


=== Generating patient for target SDP bucket: 0.55 ===

--- Hunting for Patient with SDP ≈ 0.55 ---
    [!] EXACT INFERENCE IMPOSSIBLE: Sub-network treewidth exceeds hardware limits. Skipping network.

--- Hunting for Patient with SDP ≈ 0.55 ---
    [!] EXACT INFERENCE IMPOSSIBLE: Sub-network treewidth exceeds hardware limits. Skipping network.

--- Hunting for Patient with SDP ≈ 0.55 ---
    [!] EXACT INFERENCE IMPOSSIBLE: Sub-network treewidth exceeds hardware limits. Skipping network.

--- Hunting for Patient with SDP ≈ 0.55 ---
    [!] EXACT INFERENCE IMPOSSIBLE: Sub-network treewidth exceeds hardware limits. Skipping network.

--- Hunting for Patient with SDP ≈ 0.55 ---
    [!] EXACT INFERENCE IMPOSSIBLE: Sub-network treewidth exceeds hardware limits. Skipping network.

--- Hunting for Patient with SDP ≈ 0.55 ---
    [!] EXACT INFERENCE IMPOSSIBLE: Sub-network treewidth exceeds hardware limits. Skipping network.

--- Hunting for Patient with SDP ≈ 0.55 ---
    [!] EXACT INFERENCE

In [24]:
len(patients)

0

In [25]:
patients.keys()

dict_keys([])

In [14]:
patients[0.55]

{'evidence': {'X0': '0',
  'X1': '1',
  'X11': '0',
  'X13': '1',
  'X14': '1',
  'X15': '1',
  'X16': '1',
  'X17': '1',
  'X18': '1',
  'X19': '0',
  'X20': '0',
  'X21': '1',
  'X23': '1',
  'X24': '0',
  'X25': '1',
  'X26': '0',
  'X27': '1',
  'X28': '0',
  'X29': '1',
  'X4': '1',
  'X7': '1',
  'X8': '0'},
 'true_sdp': 0.5100062698554809}

In [13]:
est_sdp = fast_mcmc_sdp_estimation(model, target, '1', patients[0.55]['evidence'], 0.5)
print(est_sdp)

0.5075454545454545


# Run the experiment

In [29]:
def harvest_patients_for_all_buckets(bn, target_node, target_value, decision_threshold, evidence_vars, target_buckets, tolerance=0.05, max_restarts=70, max_steps_per_restart=400):
    """
    Wanders the probability landscape using STRICT Stochastic Hill Climbing.
    If it gets trapped in a local minimum, it triggers a Random Restart.
    Harvests any patient that fits an empty bucket along the way.
    """
    print(f"\n--- Starting Hill Climbing Harvest for Buckets: {target_buckets} ---")
    
    # 1. Setup Pruned Sub-model for fast base-decision checks
    relevant_nodes = list(evidence_vars) + [target_node]
    ancestral_structure = bn.get_ancestral_graph(relevant_nodes)
    sub_model = BayesianNetwork(ancestral_structure.edges())
    sub_model.add_nodes_from(ancestral_structure.nodes())
    for node in sub_model.nodes():
        sub_model.add_cpds(bn.get_cpds(node))
    inference = VariableElimination(sub_model)
    
    unfilled_buckets = {bucket: None for bucket in target_buckets}
    
    # ========================================================
    # RANDOM RESTART LOOP
    # ========================================================
    for restart in range(max_restarts):
        empty_targets = [b for b, v in unfilled_buckets.items() if v is None]
        if not empty_targets:
            break # We filled them all!
            
        # 1. Find a Valid Starting Seed for this climb
        current_patient = None
        attempts = 0
        while current_patient is None and attempts < 1000:
            temp_patient = {var: random.choice(sub_model.get_cpds(var).state_names[var]) for var in evidence_vars}
            try:
                base_dist = inference.query(variables=[target_node], evidence=temp_patient, show_progress=False)
                if base_dist.get_value(**{target_node: target_value}) >= decision_threshold:
                    current_patient = temp_patient
            except (ValueError, MemoryError):
                return unfilled_buckets # Fast fail on impossible networks
            attempts += 1
            
        if current_patient is None:
            continue
            
        hidden_vars = [v for v in bn.nodes() if v not in current_patient and v != target_node]
        partitions = get_partitions(bn, hidden_vars, target_node, current_patient)
        
        try:
            current_sdp = fast_broadcast_sdp(bn, target_node, target_value, current_patient, decision_threshold, partitions)
        except (ValueError, MemoryError):
            return unfilled_buckets
            
        # Check if the random seed filled anything!
        for bucket in empty_targets:
            if abs(current_sdp - bucket) <= tolerance:
                unfilled_buckets[bucket] = (current_patient.copy(), current_sdp)
                print(f"    [+] INSTANT HARVEST (Restart {restart}): Filled bucket {bucket} with SDP {current_sdp:.4f}!")
                empty_targets = [b for b, v in unfilled_buckets.items() if v is None]
                
        if not empty_targets:
            break
            
        # ========================================================
        # HILL CLIMBING LOOP
        # ========================================================
        # We track patience. If we reject X mutations in a row, we are stuck.
        patience = len(evidence_vars) 
        stuck_counter = 0
        
        for step in range(max_steps_per_restart):
            if not empty_targets:
                break
                
            # Gravity: Pull toward the nearest empty bucket
            active_target = min(empty_targets, key=lambda b: abs(current_sdp - b))
            current_error = abs(current_sdp - active_target)
            
            # Mutate 1 random variable (Stochastic HC)
            var_to_mutate = random.choice(evidence_vars)
            possible_states = sub_model.get_cpds(var_to_mutate).state_names[var_to_mutate]
            
            proposed_patient = current_patient.copy()
            proposed_patient[var_to_mutate] = random.choice([s for s in possible_states if s != proposed_patient[var_to_mutate]])

            # Base Anchor Check
            try:
                base_dist = inference.query(variables=[target_node], evidence=proposed_patient, show_progress=False)
                if base_dist.get_value(**{target_node: target_value}) < decision_threshold:
                    stuck_counter += 1
                    if stuck_counter >= patience: break # Local minimum reached
                    continue 
            except (ValueError, MemoryError):
                stuck_counter += 1
                continue 
                
            # Evaluate Exact SDP
            partitions = get_partitions(bn, hidden_vars, target_node, proposed_patient)
            try:
                proposed_sdp = fast_broadcast_sdp(bn, target_node, target_value, proposed_patient, decision_threshold, partitions)
            except (ValueError, MemoryError):
                stuck_counter += 1
                continue 
                
            # --- THE HARVEST CHECK ---
            for bucket in empty_targets:
                if abs(proposed_sdp - bucket) <= tolerance:
                    unfilled_buckets[bucket] = (proposed_patient.copy(), proposed_sdp)
                    print(f"    [+] HARVEST SUCCESS (Restart {restart}, Step {step}): Filled bucket {bucket} (Exact SDP: {proposed_sdp:.4f})!")
                    empty_targets = [b for b, v in unfilled_buckets.items() if v is None]
            
            if not empty_targets:
                break
                
            # --- STRICT GREEDY ACCEPTANCE ---
            proposed_error = abs(proposed_sdp - active_target)
            
            if proposed_error < current_error:
                # We moved closer! Accept and reset the stuck counter.
                current_patient = proposed_patient
                current_sdp = proposed_sdp
                stuck_counter = 0
            else:
                # We got worse (or hit a flat plateau). Reject it.
                stuck_counter += 1
                
            # If we reject too many times in a row, we are trapped
            # Break the inner loop to trigger a Random Restart!
            if stuck_counter >= patience:
                # print(f"      -> Trapped in local minimum at SDP {current_sdp:.4f}. Restarting...")
                break 
                
    remaining = [b for b, v in unfilled_buckets.items() if v is None]
    if remaining:
        print(f"--- Harvest Complete. Could not fill buckets: {remaining} ---")
    else:
        print(f"--- Harvest Complete. All buckets filled successfully! ---")
        
    return unfilled_buckets

In [30]:
buckets = harvest_patients_for_all_buckets(model, target, '1', 0.5, evidence_vars, [0.5, 0.7, 0.9])


--- Starting Hill Climbing Harvest for Buckets: [0.5, 0.7, 0.9] ---


In [31]:
buckets

{0.5: None, 0.7: None, 0.9: None}

In [ ]:
def run_targeted_sdp_experiment(bif_directory, output_csv="targeted_sdp_benchmark.csv"):
    bif_files = glob.glob(os.path.join(bif_directory, "*.bif"))
    results = []
    
    H_RATIO = 0.25
    DECISION_THRESHOLD = 0.5
    TARGET_BUCKETS = [0.50, 0.60, 0.70, 0.80, 0.90, 1.0]
    MCMC_TRIALS = 10
    
    for file in bif_files:
        n_nodes, density, rigidity = parse_bn_filename(file)
        print(f"\n========================================")
        print(f"Loading: {os.path.basename(file)}")
        
        bn = BIFReader(file).get_model()
        all_nodes = list(bn.nodes())
        
        target = select_optimal_target_node(bn)
        target_states = bn.get_cpds(target).state_names[target]
        target_value = target_states[1] if len(target_states) > 1 else target_states[0]
        
        available_nodes = [n for n in all_nodes if n != target]
        n_hidden = max(1, int(len(available_nodes) * H_RATIO))
        hidden_vars = random.sample(available_nodes, n_hidden)
        evidence_vars = [n for n in available_nodes if n not in hidden_vars]
        
        # Run the Harvester 
        harvested_data = harvest_patients_for_all_buckets(
            bn, target, target_value, DECISION_THRESHOLD, evidence_vars, TARGET_BUCKETS
        )
        
        # Now process whatever it managed to find
        for target_sdp, result in harvested_data.items():
            if result is None:
                continue # We didn't find a patient for this specific bucket in this network
                
            patient, exact_sdp = result
            print(f"\n  -> Benchmarking found patient for bucket {target_sdp} (Exact: {exact_sdp:.4f})")
            
            # ========================================================
            # RACE TIMING 1: EXACT SDP
            # ========================================================
            partitions = get_partitions(bn, hidden_vars, target, patient)
            exact_time = np.nan
            
            try:
                start_time = time.time()
                # Re-run the exact calculation once just to time it cleanly
                exact_sdp_benchmark = fast_broadcast_sdp(bn, target, target_value, patient, DECISION_THRESHOLD, partitions)
                exact_time = time.time() - start_time
                print(f"       -> Exact Time: {exact_time:.4f} seconds")
            except (ValueError, MemoryError):
                print(f"       -> Exact Time: [FAILED DUE TO MEMORY/EINSUM LIMIT]")
            
            # ========================================================
            # RACE TIMING 2: MCMC SDP
            # ========================================================
            mcmc_estimates = []
            mcmc_times = []
            
            for trial in range(MCMC_TRIALS):
                start_time = time.time()
                est_sdp = fast_mcmc_sdp_estimation(
                    bn, target, target_value, patient, DECISION_THRESHOLD,
                    n_samples=7000, burn_in=500, thinning=5
                )
                mcmc_times.append(time.time() - start_time)
                mcmc_estimates.append(est_sdp)
                
            mcmc_mean = np.mean(mcmc_estimates)
            mcmc_variance = np.var(mcmc_estimates)
            mcmc_avg_time = np.mean(mcmc_times)
            
            print(f"       -> MCMC Avg Time: {mcmc_avg_time:.4f} seconds")
            
            absolute_error = abs(exact_sdp - mcmc_mean)
            
            # Record everything to the dataset
            results.append({
                'Network': os.path.basename(file),
                'N_Nodes': n_nodes,
                'Density': density,
                'Rigidity': rigidity,
                'Target_Bucket': target_sdp,
                'Target_Node': target,
                'Target_Value': target_value,
                'Exact_SDP': exact_sdp,
                'Exact_Time_sec': exact_time,
                'MCMC_Mean_SDP': mcmc_mean,
                'MCMC_Variance': mcmc_variance,
                'MCMC_Avg_Time_sec': mcmc_avg_time,
                'Absolute_Error': absolute_error
            })
            
            # Save progressively
            pd.DataFrame(results).to_csv(output_csv, index=False)

    print(f"\nExperiment Complete! Results saved to {output_csv}")
    return pd.DataFrame(results)

In [33]:
print('='*100)

In [34]:
run_targeted_sdp_experiment('/home/joao/Desktop/UFMG/PhD/code/explaining-BN/src/generated_bif_files')


Loading: bn_n50_w12_dense_fuzzy.bif

--- Starting Hill Climbing Harvest for Buckets: [0.5, 0.6, 0.7, 0.8, 0.9, 1.0] ---
    [+] INSTANT HARVEST (Restart 0): Filled bucket 0.8 with SDP 0.8239!
    [+] HARVEST SUCCESS (Restart 0, Step 3): Filled bucket 0.7 (Exact SDP: 0.6800)!
    [+] HARVEST SUCCESS (Restart 0, Step 4): Filled bucket 0.9 (Exact SDP: 0.8638)!
    [+] HARVEST SUCCESS (Restart 0, Step 6): Filled bucket 1.0 (Exact SDP: 0.9612)!
    [+] HARVEST SUCCESS (Restart 0, Step 16): Filled bucket 0.6 (Exact SDP: 0.5899)!
    [+] HARVEST SUCCESS (Restart 1, Step 7): Filled bucket 0.5 (Exact SDP: 0.4903)!
--- Harvest Complete. All buckets filled successfully! ---

  -> Benchmarking found patient for bucket 0.5 (Exact: 0.4903)
       -> Exact Time: 0.0338 seconds
       -> MCMC Avg Time: 12.4671 seconds

  -> Benchmarking found patient for bucket 0.6 (Exact: 0.5899)
       -> Exact Time: 0.0319 seconds
       -> MCMC Avg Time: 12.1370 seconds

  -> Benchmarking found patient for bucket

/home/joao/anaconda3/envs/bn-medical/lib/python3.8/site-packages/pgmpy/factors/discrete/DiscreteFactor.py:478: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / phi.values.sum()


    [+] HARVEST SUCCESS (Restart 2, Step 38): Filled bucket 0.5 (Exact SDP: 0.5099)!
--- Harvest Complete. All buckets filled successfully! ---

  -> Benchmarking found patient for bucket 0.5 (Exact: 0.5099)
       -> Exact Time: 0.0305 seconds
       -> MCMC Avg Time: 11.9958 seconds

  -> Benchmarking found patient for bucket 0.6 (Exact: 0.5923)
       -> Exact Time: 0.0275 seconds
       -> MCMC Avg Time: 12.0699 seconds

  -> Benchmarking found patient for bucket 0.7 (Exact: 0.7333)
       -> Exact Time: 0.0273 seconds
       -> MCMC Avg Time: 11.9086 seconds

  -> Benchmarking found patient for bucket 0.8 (Exact: 0.8215)
       -> Exact Time: 0.0294 seconds
       -> MCMC Avg Time: 11.9836 seconds

  -> Benchmarking found patient for bucket 0.9 (Exact: 0.9493)
       -> Exact Time: 0.0284 seconds
       -> MCMC Avg Time: 12.0975 seconds

  -> Benchmarking found patient for bucket 1.0 (Exact: 1.0000)
       -> Exact Time: 0.0295 seconds
       -> MCMC Avg Time: 11.9256 seconds

Loa

,Network,N_Nodes,Density,Rigidity,Target_Bucket,Target_Node,Target_Value,Exact_SDP,Exact_Time_sec,MCMC_Mean_SDP,MCMC_Variance,MCMC_Avg_Time_sec,Absolute_Error
0,bn_n50_w12_dense_fuzzy.bif,50,dense,fuzzy,0.5,X20,1,0.490288,0.033820,0.488800,0.000141,12.467072,0.001488
1,bn_n50_w12_dense_fuzzy.bif,50,dense,fuzzy,0.6,X20,1,0.589934,0.031909,0.589943,0.000161,12.137021,0.000009
2,bn_n50_w12_dense_fuzzy.bif,50,dense,fuzzy,0.7,X20,1,0.679995,0.032421,0.682343,0.000062,12.042528,0.002348
3,bn_n50_w12_dense_fuzzy.bif,50,dense,fuzzy,0.8,X20,1,0.823873,0.034249,0.826543,0.000035,12.011616,0.002670
4,bn_n50_w12_dense_fuzzy.bif,50,dense,fuzzy,0.9,X20,1,0.863764,0.032459,0.862929,0.000034,12.135919,0.000836
...,...,...,...,...,...,...,...,...,...,...,...,...,...
192,bn_n40_w2_sparse_det.bif,40,sparse,det,0.6,X14,1,0.640963,0.018320,0.749657,0.019627,6.067269,0.108694
193,bn_n40_w2_sparse_det.bif,40,sparse,det,0.7,X14,1,0.721909,0.014230,0.728443,0.013209,5.926464,0.006534
194,bn_n40_w2_sparse_det.bif,40,sparse,det,0.8,X14,1,0.811625,0.014888,0.649357,0.053406,6.000893,0.162268
195,bn_n40_w2_sparse_det.bif,40,sparse,det,0.9,X14,1,0.856314,0.013380,0.859186,0.000101,6.109194,0.002871
